# BunaVision — Coffee Domain Validator Training

Trains a **MobileNetV3-Small binary classifier** (`coffee` / `not_coffee`) as a domain gate
for the BunaVision defect detection pipeline.

**Output:** `best_validator.onnx` — drop this into `vision/model/` in the repo.


---

### How data is collected
- **Positives (coffee):** YOLO training images from Robflow
- **Negatives (not_coffee):** Streamed one-by-one from the COCO 2017 validation set via HuggingFace.
  Only the images you actually iterate over are downloaded.

## 0 — Runtime check
Make sure you are on a GPU runtime: `Runtime → Change runtime type → T4 GPU`

In [4]:
import torch
assert torch.cuda.is_available(), "Switch to GPU runtime first (Runtime → Change runtime type → T4 GPU)"
print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU: Tesla T4


## 1 — Install dependencies

In [5]:
!pip install -q datasets onnx onnxruntime onnxscript Pillow torchvision roboflow

## 2 — Download coffee images via Roboflow (positives)

This downloads the same Roboflow dataset used to train the YOLO defect detector.
The training split (1,444 images) becomes the positive class for the binary classifier.

Enter your Roboflow API key below

In [6]:

ROBOFLOW_API_KEY = ""   # Enter your Roboflow API key

assert ROBOFLOW_API_KEY, "Set your ROBOFLOW_API_KEY above"

from roboflow import Roboflow
from pathlib import Path

rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('lulus-vpibo').project('green-coffee-bean-defects')
dataset = project.version(14).download('yolov8')

# After download, images live at: /content/green-coffee-bean-defects-14/train/images/
COFFEE_IMAGES_DIR = Path(dataset.location) / 'train' / 'images'
assert COFFEE_IMAGES_DIR.exists(), f"Expected images at {COFFEE_IMAGES_DIR}"
print(f"Coffee images directory: {COFFEE_IMAGES_DIR}")
print(f"Available images: {len(list(COFFEE_IMAGES_DIR.glob('*.jpg')))}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to green-coffee-bean-defects-14 in yolov8:: 100%|██████████| 3460/3460 [00:01<00:00, 1916.76it/s]

Coffee images directory: /content/green-coffee-bean-defects-14/train/images
Available images: 1444


## 3 — Collect positives (coffee images)

In [7]:
import shutil
from pathlib import Path

MAX_POSITIVES = 500  # cap to keep classes balanced

pos_dir = Path('/content/data/coffee')
pos_dir.mkdir(parents=True, exist_ok=True)

source_images = sorted(COFFEE_IMAGES_DIR.glob('*.jpg')) + \
                sorted(COFFEE_IMAGES_DIR.glob('*.png'))
source_images = source_images[:MAX_POSITIVES]

for img_path in source_images:
    shutil.copy(img_path, pos_dir / img_path.name)

print(f"Positive images collected: {len(list(pos_dir.iterdir()))}")

Positive images collected: 500


## 4 — Stream negatives from COCO via HuggingFace

No full download — HuggingFace `streaming=True` fetches each image on demand.
Only `NUM_NEGATIVES` images are ever pulled.

In [8]:
from datasets import load_dataset
from PIL import Image
from pathlib import Path

NUM_NEGATIVES = 500

neg_dir = Path('/content/data/not_coffee')
neg_dir.mkdir(parents=True, exist_ok=True)

# Streams COCO 2017 validation — only downloads images you iterate over.
# No credentials needed.
ds = load_dataset(
    'detection-datasets/coco',
    split='val',
    streaming=True,
    trust_remote_code=True,
)

count = 0
for sample in ds:
    if count >= NUM_NEGATIVES:
        break
    img: Image.Image = sample['image']
    # Convert palette/RGBA images to RGB
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img.save(neg_dir / f'neg_{count:04d}.jpg', quality=90)
    count += 1
    if count % 100 == 0:
        print(f"  Streamed {count}/{NUM_NEGATIVES} negatives...")

print(f"Negative images collected: {len(list(neg_dir.iterdir()))}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'detection-datasets/coco' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'detection-datasets/coco' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

dataset_infos.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

  Streamed 100/500 negatives...
  Streamed 200/500 negatives...
  Streamed 300/500 negatives...
  Streamed 400/500 negatives...
  Streamed 500/500 negatives...
Negative images collected: 500


## 5 — Build train / val splits

In [9]:
import random
import shutil
from pathlib import Path

SEED = 42
VAL_RATIO = 0.2

random.seed(SEED)

def split_class(src_dir: Path, class_name: str, val_ratio: float):
    images = list(src_dir.glob('*.jpg')) + list(src_dir.glob('*.png'))
    random.shuffle(images)
    split = int(len(images) * (1 - val_ratio))
    train_imgs, val_imgs = images[:split], images[split:]

    for subset, imgs in [('train', train_imgs), ('val', val_imgs)]:
        dest = Path(f'/content/split/{subset}/{class_name}')
        dest.mkdir(parents=True, exist_ok=True)
        for img in imgs:
            shutil.copy(img, dest / img.name)

    print(f"  {class_name}: {len(train_imgs)} train / {len(val_imgs)} val")

split_class(Path('/content/data/coffee'), 'coffee', VAL_RATIO)
split_class(Path('/content/data/not_coffee'), 'not_coffee', VAL_RATIO)

  coffee: 400 train / 100 val
  not_coffee: 400 train / 100 val


## 6 — Dataset + DataLoader

In [10]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# MobileNetV3-Small expects 224x224, ImageNet normalisation
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder('/content/split/train', transform=train_tf)
val_ds   = datasets.ImageFolder('/content/split/val',   transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# class_to_idx tells us which index maps to 'coffee'
print(f"Class mapping: {train_ds.class_to_idx}")
print(f"Train: {len(train_ds)} images | Val: {len(val_ds)} images")

Class mapping: {'coffee': 0, 'not_coffee': 1}
Train: 800 images | Val: 200 images


## 7 — Build model

MobileNetV3-Small pretrained on ImageNet. We freeze the backbone and only train the
classifier head for the first few epochs, then unfreeze everything for fine-tuning.

In [11]:
import torch.nn as nn
from torchvision import models

device = torch.device('cuda')

model = models.mobilenet_v3_small(
    weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1
)

# Replace the final classifier layer (originally 1000 ImageNet classes → 2 classes)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, 2)

# Freeze backbone — only train the new head first
for param in model.features.parameters():
    param.requires_grad = False

model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 109MB/s]


Total params: 1,519,906 | Trainable: 592,898


## 8 — Train

In [12]:
import torch.optim as optim

# Phase 1: train only the head (5 epochs)
PHASE1_EPOCHS = 5
# Phase 2: unfreeze all, fine-tune with lower LR (10 epochs)
PHASE2_EPOCHS = 10

criterion = nn.CrossEntropyLoss()

def run_epoch(loader, model, criterion, optimizer=None):
    """Run one epoch. If optimizer is None, runs in eval mode."""
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()

    with ctx:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)

    return total_loss / total, correct / total


# Phase 1: head only
print("Phase 1 — training classifier head only")
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

for epoch in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, model, criterion, optimizer)
    va_loss, va_acc = run_epoch(val_loader,   model, criterion)
    print(f"  Epoch {epoch:02d}/{PHASE1_EPOCHS}  "
          f"train loss={tr_loss:.4f} acc={tr_acc:.3f}  "
          f"val loss={va_loss:.4f} acc={va_acc:.3f}")

#  Phase 2: full fine-tune
print("\nPhase 2 — fine-tuning all layers")
for param in model.features.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

best_val_acc = 0.0
best_state   = None

for epoch in range(1, PHASE2_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, model, criterion, optimizer)
    va_loss, va_acc = run_epoch(val_loader,   model, criterion)
    scheduler.step()
    marker = ' ← best' if va_acc > best_val_acc else ''
    print(f"  Epoch {epoch:02d}/{PHASE2_EPOCHS}  "
          f"train loss={tr_loss:.4f} acc={tr_acc:.3f}  "
          f"val loss={va_loss:.4f} acc={va_acc:.3f}{marker}")
    if va_acc > best_val_acc:
        best_val_acc = va_acc
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}

# Restore best checkpoint
model.load_state_dict(best_state)
print(f"\nBest val accuracy: {best_val_acc:.3f}")

Phase 1 — training classifier head only
  Epoch 01/5  train loss=0.0475 acc=0.985  val loss=0.5783 acc=0.855
  Epoch 02/5  train loss=0.0066 acc=0.998  val loss=0.0562 acc=0.980
  Epoch 03/5  train loss=0.0011 acc=1.000  val loss=0.0157 acc=0.990
  Epoch 04/5  train loss=0.0005 acc=1.000  val loss=0.0083 acc=1.000
  Epoch 05/5  train loss=0.0015 acc=1.000  val loss=0.0002 acc=1.000

Phase 2 — fine-tuning all layers
  Epoch 01/10  train loss=0.0003 acc=1.000  val loss=0.0001 acc=1.000 ← best
  Epoch 02/10  train loss=0.0001 acc=1.000  val loss=0.0000 acc=1.000
  Epoch 03/10  train loss=0.0007 acc=1.000  val loss=0.0000 acc=1.000
  Epoch 04/10  train loss=0.0000 acc=1.000  val loss=0.0000 acc=1.000
  Epoch 05/10  train loss=0.0000 acc=1.000  val loss=0.0000 acc=1.000
  Epoch 06/10  train loss=0.0003 acc=1.000  val loss=0.0000 acc=1.000
  Epoch 07/10  train loss=0.0021 acc=0.999  val loss=0.0000 acc=1.000
  Epoch 08/10  train loss=0.0000 acc=1.000  val loss=0.0000 acc=1.000
  Epoch 09/10 

## 9 — Evaluate on validation set

Print per-class accuracy and a confusion matrix.

In [13]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        logits = model(images.to(device))
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

class_names = [k for k, _ in sorted(train_ds.class_to_idx.items(), key=lambda x: x[1])]
print(classification_report(all_labels, all_preds, target_names=class_names))
print("Confusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(all_labels, all_preds))

              precision    recall  f1-score   support

      coffee       1.00      1.00      1.00       100
  not_coffee       1.00      1.00      1.00       100

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

Confusion matrix (rows=actual, cols=predicted):
[[100   0]
 [  0 100]]


## 10 — Export to ONNX

Exports the best checkpoint as `best_validator.onnx`.
Also saves `validator_config.json` so the service knows which index is `coffee`.

In [14]:
import json
import onnx

ONNX_PATH   = '/content/best_validator.onnx'
CONFIG_PATH = '/content/validator_config.json'

# Export
model.eval().cpu()
dummy = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    dummy,
    ONNX_PATH,
    opset_version=18,
    input_names=['input'],
    output_names=['logits'],
    dynamic_axes={'input': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},
    dynamo=False,
)

# Verify the exported model
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)

# Save config
class_to_idx = train_ds.class_to_idx  # e.g. {'coffee': 0, 'not_coffee': 1} — order may vary
coffee_idx   = class_to_idx['coffee']

config = {
    "model_version": "MobileNetV3-Small-Coffee-ONNX-v1",
    "labels": [k for k, _ in sorted(class_to_idx.items(), key=lambda x: x[1])],
    "coffee_class_index": coffee_idx,
    "confidence_threshold": 0.75,
    "input_size": [224, 224],
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std":  [0.229, 0.224, 0.225]
}
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=2)

import os
size_mb = os.path.getsize(ONNX_PATH) / 1024 / 1024
print(f"ONNX model exported: {ONNX_PATH} ({size_mb:.1f} MB)")
print(f"Config saved:        {CONFIG_PATH}")
print(f"coffee_class_index:  {coffee_idx}  (verify this matches the label order)")
print(f"class_to_idx:        {class_to_idx}")

You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html


ONNX model exported: /content/best_validator.onnx (5.8 MB)
Config saved:        /content/validator_config.json
coffee_class_index:  0  (verify this matches the label order)
class_to_idx:        {'coffee': 0, 'not_coffee': 1}


## 11 — Sanity-check with ONNX Runtime

Runs one coffee image and one negative through the exported model to confirm it works.

In [15]:
import onnxruntime as ort
import numpy as np
from PIL import Image
from torchvision import transforms

sess = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def predict(img_path: str) -> dict:
    img    = Image.open(img_path).convert('RGB')
    tensor = preprocess(img).unsqueeze(0).numpy()  # (1, 3, 224, 224)
    logits = sess.run(['logits'], {'input': tensor})[0]  # (1, 2)
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()[0]
    return {
        'coffee_prob':     float(probs[coffee_idx]),
        'not_coffee_prob': float(probs[1 - coffee_idx]),
        'prediction':      'coffee' if probs[coffee_idx] >= 0.75 else 'not_coffee',
    }

# Test with one positive and one negative
coffee_sample  = next(Path('/content/split/val/coffee').glob('*.jpg'))
neg_sample     = next(Path('/content/split/val/not_coffee').glob('*.jpg'))

print("Coffee image:    ", predict(str(coffee_sample)))
print("Not-coffee image:", predict(str(neg_sample)))

Coffee image:     {'coffee_prob': 1.0, 'not_coffee_prob': 4.423945931808504e-11, 'prediction': 'coffee'}
Not-coffee image: {'coffee_prob': 4.351888294173989e-11, 'not_coffee_prob': 1.0, 'prediction': 'not_coffee'}


## 12 — Save and Download Model Artifacts


In [16]:
import shutil
from pathlib import Path
from google.colab import files

OUTPUT_DRIVE_DIR = Path('/content/drive/MyDrive/BunaVision/validator_output')
if Path('/content/drive/MyDrive').exists():
    OUTPUT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(ONNX_PATH, OUTPUT_DRIVE_DIR / 'best_validator.onnx')
    shutil.copy(CONFIG_PATH, OUTPUT_DRIVE_DIR / 'validator_config.json')
    print(f"Artifacts saved to Google Drive: {OUTPUT_DRIVE_DIR}")

# Direct download to local machine
files.download(ONNX_PATH)
files.download(CONFIG_PATH)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>